### Operational Supervisor

Creates a single Multi-Agent Supervisor (MAS) for the operational dashboard that
routes across **3 Genie spaces** and **6 Knowledge Assistants**:

| Sub-agent | Type | Source |
|---|---|---|
| `revenue-analytics` | Genie | Revenue & Orders Intelligence space |
| `operations-intelligence` | Genie | Operations Intelligence space |
| `menu-analytics` | Genie | Menu & Safety Intelligence space |
| `inspection-reports` | KA | Food safety inspection PDFs |
| `menu-document-search` | KA | Restaurant menu PDFs |
| `legal-complaints` | KA | Legal complaint case files |
| `regulatory-compliance` | KA | Permits, certifications, FDA |
| `audit-findings` | KA | Financial / operational / supply-chain audits |
| `consultancy-strategy` | KA | Strategic consulting reports |

In [ ]:
%pip install --upgrade databricks-sdk mlflow-skinny[databricks]

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG             = dbutils.widgets.get("CATALOG")
SUPERVISOR_ENDPOINT = dbutils.widgets.get("SUPERVISOR_ENDPOINT_NAME")
# KA endpoint names are auto-generated by the KA v2.1 API and resolved from
# uc_state below. They are no longer passed in as Job parameters.

In [ ]:
from databricks.sdk import WorkspaceClient
import json, sys, time

sys.path.append('../utils')
from uc_state import add

w = WorkspaceClient()
API_BASE = "/api/2.0/multi-agent-supervisors"

##### Resolve Genie space IDs from uc_state

Titles must match the strings used by `stages/genie_spaces.ipynb`.

In [ ]:
GENIE_TITLES = {
    "revenue": f"Revenue & Orders Intelligence ({CATALOG})",
    "ops":     f"Operations Intelligence ({CATALOG})",
    "menu":    f"Menu & Safety Intelligence ({CATALOG})",
}
genie_ids = {}

df = spark.sql(f"""
    SELECT resource_data FROM {CATALOG}._internal_state.resources
    WHERE resource_type = 'genie_spaces'
    ORDER BY created_at DESC
""")
for row in df.collect():
    info = json.loads(row.resource_data)
    title = info.get("title")
    for key, expected in GENIE_TITLES.items():
        if title == expected and key not in genie_ids:
            genie_ids[key] = info.get("space_id")

missing = [k for k in GENIE_TITLES if k not in genie_ids]
if missing:
    raise RuntimeError(
        f"Missing Genie space(s) {missing} in uc_state. Run the genie_spaces stage first."
    )

revenue_genie_id = genie_ids["revenue"]
ops_genie_id     = genie_ids["ops"]
menu_genie_id    = genie_ids["menu"]
print(f"Revenue Genie:   {revenue_genie_id}")
print(f"Operations Genie:{ops_genie_id}")
print(f"Menu Genie:      {menu_genie_id}")

##### Resolve KA serving endpoint names from uc_state

In [ ]:
KA_NAMES = [
    f"{CATALOG}-inspection-knowledge",
    f"{CATALOG}-menu-knowledge",
    f"{CATALOG}-legal",
    f"{CATALOG}-regulatory",
    f"{CATALOG}-audits",
    f"{CATALOG}-consultancy",
]


def _ka_endpoint_from_tile_id(tile_id: str) -> str:
    # Databricks serving endpoint provisioned for a KA is named
    # ka-{first-8-chars-of-tile_id}-endpoint. The `endpoint_name` field
    # returned by the v2.1 KA API is a different (display-style) value
    # and MUST NOT be used here — MAS validates the serving-endpoint name.
    if not tile_id or len(tile_id) < 8:
        raise ValueError(f"Invalid KA tile_id for endpoint construction: {tile_id!r}")
    return f"ka-{tile_id[:8]}-endpoint"


ka_tile_ids = {}

try:
    df = spark.sql(f"""
        SELECT resource_data FROM {CATALOG}._internal_state.resources
        WHERE resource_type = 'knowledge_assistants'
        ORDER BY created_at DESC
    """)
    for row in df.collect():
        info = json.loads(row.resource_data)
        name = info.get("name", "")
        tid  = info.get("tile_id", "")
        if name in KA_NAMES and tid and name not in ka_tile_ids:
            ka_tile_ids[name] = tid
    print(f"uc_state: resolved {len(ka_tile_ids)}/{len(KA_NAMES)} KA tile_ids")
except Exception as e:
    print(f"\u26a0\ufe0f uc_state lookup failed: {e}")

missing_names = [n for n in KA_NAMES if n not in ka_tile_ids]
if missing_names:
    print(f"{len(missing_names)} KA tile_id(s) missing from uc_state \u2014 listing KAs via v2.1 API...")
    ka_params = {}
    while True:
        resp = w.api_client.do("GET", "/api/2.1/knowledge-assistants", query=ka_params)
        for ka in resp.get("knowledge_assistants", []):
            dn  = ka.get("display_name", "")
            kid = ka.get("id", "")
            if dn in missing_names and kid:
                ka_tile_ids[dn] = kid
                missing_names.remove(dn)
        token = resp.get("next_page_token")
        if not token or not missing_names:
            break
        ka_params = {"page_token": token}

if missing_names:
    raise RuntimeError(
        f"Could not resolve tile_ids for: {missing_names}. "
        f"Ensure the knowledge_agents stage ran first."
    )

ka_endpoints = {name: _ka_endpoint_from_tile_id(tid) for name, tid in ka_tile_ids.items()}

inspect_ep_id = ka_endpoints[f"{CATALOG}-inspection-knowledge"]
menu_ep_id    = ka_endpoints[f"{CATALOG}-menu-knowledge"]
legal_ep_id   = ka_endpoints[f"{CATALOG}-legal"]
reg_ep_id     = ka_endpoints[f"{CATALOG}-regulatory"]
audit_ep_id   = ka_endpoints[f"{CATALOG}-audits"]
consult_ep_id = ka_endpoints[f"{CATALOG}-consultancy"]

print(f"Inspection KA endpoint:  {inspect_ep_id}")
print(f"Menu KA endpoint:        {menu_ep_id}")
print(f"Legal KA endpoint:       {legal_ep_id}")
print(f"Regulatory KA endpoint:  {reg_ep_id}")
print(f"Audit KA endpoint:       {audit_ep_id}")
print(f"Consultancy KA endpoint: {consult_ep_id}")

##### Create the Operational Supervisor

In [ ]:
AGENT_NAME = f"{CATALOG}-operational-supervisor"

EXAMPLES = [
    {
        "question": "Give me the executive briefing on the state of the business",
        "guideline": "Should summarize revenue performance, top operational risks, any critical legal or compliance issues, and one strategic recommendation. Keep it to 5 bullet points maximum.",
    },
    {
        "question": "Which location is most at risk right now?",
        "guideline": "Should consider complaint rate, inspection score, legal exposure, and audit findings for each location. Rank and explain the top risk with specific numbers.",
    },
    {
        "question": "Which location has the highest order cancellation rate right now?",
        "guideline": "Must name exactly one specific location with the highest rate. Must include the numeric cancellation rate as a percentage. Routes to revenue-analytics or operations-intelligence.",
    },
    {
        "question": "How does revenue compare across our locations this week?",
        "guideline": "Must provide revenue figures or ranking for multiple locations. Must reference the current week. Routes to revenue-analytics.",
    },
    {
        "question": "Which brand is generating the most revenue right now?",
        "guideline": "Must name a specific brand (not a location). Must include a revenue figure or ranking. Routes to revenue-analytics.",
    },
    {
        "question": "Which location needs the most operational attention right now?",
        "guideline": "Must name one specific location with the highest operational risk. Must justify with at least two operational metrics. Routes to operations-intelligence.",
    },
    {
        "question": "What happened during the Chicago food safety inspection? Were there critical violations?",
        "guideline": "Must reference a specific inspection report by date or ID. Must state the inspection score and/or grade. Must explicitly state whether critical violations were found and cite at least one specific violation code if they exist. Routes to inspection-reports.",
    },
    {
        "question": "Which menu items are gluten-free and under $15?",
        "guideline": "Must list specific items with brand, price, and allergen status. Routes to menu-analytics or menu-document-search.",
    },
    {
        "question": "Compare protein content across all burgers in our network",
        "guideline": "Must return per-item protein values across brands. Routes to menu-analytics.",
    },
    {
        "question": "Do we have any active high-risk legal cases? What is the total financial exposure?",
        "guideline": "Must confirm whether active cases exist, cite at least one specific case number (CK-XX-XXXX), include a risk classification, state a financial exposure amount, and remind users to involve legal counsel. Routes to legal-complaints.",
    },
    {
        "question": "Are there any permits or regulatory certificates expiring in the next 60 days?",
        "guideline": "Must list specific document IDs or permit names with expiry dates and locations. Routes to regulatory-compliance.",
    },
    {
        "question": "What were the most significant audit findings this quarter?",
        "guideline": "Must cite the auditing firm and audit period, classify findings by severity, and state remediation status. Routes to audit-findings.",
    },
    {
        "question": "What do our consultants recommend as the top AI investments for the next 90 days?",
        "guideline": "Must reference a specific consulting report, include at least one concrete recommendation with a financial metric, and frame in the 90-day horizon. Routes to consultancy-strategy.",
    },
    {
        "question": "Give me a board deck summary: revenue performance, top operational risk, legal exposure, and one strategic recommendation",
        "guideline": "Must explicitly address all four domains. Must include at least one concrete number per domain. Must be structured with clear sections. Routes to multiple agents.",
    },
]

# ─────────────────────────────────────────────────────────────────────────────
# Source-of-truth supervisor instructions live here as a literal so this
# notebook stays self-contained and the Prompt Registry cell below can extract
# them on every deploy. At runtime we try to load the latest production
# version from the Prompt Registry first — that lets a manually edited prompt
# (Catalog Explorer → Prompts → bump @production alias) take effect on the
# NEXT deploy of this stage without having to edit this notebook.
#
# Caveats:
#   • This is "construction-time" load_prompt — the MAS service itself does
#     not call load_prompt. Editing the prompt in UC has no effect until the
#     next stage redeploy rebuilds SUPERVISOR_BODY.
#   • The Prompt Registry cell below re-registers _FALLBACK_... on every
#     deploy and bumps the @production alias to the new version, so manual
#     UC edits get clobbered on the NEXT redeploy. Backport edits to this
#     literal to make them stick.
_FALLBACK_SUPERVISOR_INSTRUCTIONS = (
    "You are an elite operational AI assistant for Casper's Kitchens, a ghost kitchen network "
    "operating 16 restaurant brands across 8 locations (US: San Francisco, Silicon Valley, Bellevue, "
    "Chicago; EMEA: London, Munich, Amsterdam, Vianen). Synthesise intelligence from multiple "
    "sources to deliver concise, executive-ready insights. Always specify which location(s) you "
    "are referencing. Lead with the most important finding, then provide supporting detail.\n\n"
    "DATA SOURCE BOUNDARIES:\n"
    "(1) 'Legal issues', 'lawsuits', 'legal cases', 'legal exposure' → legal-complaints agent "
    "(CK-XX-XXXX case numbers), NOT food safety violations or operational complaints.\n"
    "(2) 'Food safety violations' → operations-intelligence (counts/grades) or inspection-reports "
    "(detailed findings) — they are compliance issues, not lawsuits.\n"
    "(3) 'Customer complaints' (complaint rate, cancel rate) → operations-intelligence — they are "
    "operational metrics, not legal filings.\n"
    "(4) 'Allergens / nutrition / prices' → menu-analytics (structured) or menu-document-search "
    "(descriptive dish details).\n"
    "Never conflate food safety violations or operational complaint counts with legal case filings. "
    "When discussing legal or audit matters, remind users to involve counsel or auditors for "
    "decisions. Be direct, data-driven, and strategic."
)

import mlflow
import mlflow.genai

# Required: without databricks-uc the registry client hits workspace MLflow,
# where 3-part names are opaque strings and prompts never land in UC.
mlflow.set_registry_uri("databricks-uc")

_SUPERVISOR_PROMPT_URI = f"prompts:/{CATALOG}.prompts.supervisor_instructions@production"
try:
    _supervisor_instructions = mlflow.genai.load_prompt(_SUPERVISOR_PROMPT_URI).template
    print(f"[supervisor] Loaded instructions from {_SUPERVISOR_PROMPT_URI}")
except Exception as _exc:
    print(
        f"[supervisor] Could not load {_SUPERVISOR_PROMPT_URI} "
        f"({type(_exc).__name__}: {_exc}); using _FALLBACK_SUPERVISOR_INSTRUCTIONS."
    )
    _supervisor_instructions = _FALLBACK_SUPERVISOR_INSTRUCTIONS

SUPERVISOR_BODY = {
    "name": AGENT_NAME,
    "description": (
        "Operational AI assistant for Casper's Kitchens. Synthesises intelligence across revenue "
        "analytics, operations, food safety, menu and nutrition data, legal exposure, regulatory "
        "compliance, audit findings, and strategic consulting across all 8 ghost kitchen locations "
        "(US: San Francisco, Silicon Valley, Bellevue, Chicago; EMEA: London, Munich, Amsterdam, "
        "Vianen) and 16 restaurant brands."
    ),
    "endpoint_name": SUPERVISOR_ENDPOINT,
    "agents": [
        {
            "agent_type": "genie",
            "genie_space": {"id": revenue_genie_id},
            "name": "revenue-analytics",
            "description": (
                "ONLY call for questions about: revenue ($), sales totals, order counts, average order "
                "value, brand sales rankings, location revenue comparisons, or financial performance "
                "metrics. Keywords: revenue, sales, earned, orders, top performing, best/worst location "
                "by revenue, avg order, financial. Do NOT call for food safety, inspections, legal, "
                "audits, menus, or strategic questions."
            ),
        },
        {
            "agent_type": "genie",
            "genie_space": {"id": ops_genie_id},
            "name": "operations-intelligence",
            "description": (
                "ONLY call for questions about: order throughput, kitchen operations, delivery performance, "
                "food safety inspection scores, food safety violation counts, location health metrics, "
                "or operational risk. Keywords: operations, kitchen, delivery, inspection score, food "
                "safety grade, throughput, busiest hours, peak demand, operational issues, cancel rate, "
                "complaint rate. Do NOT call for revenue totals, legal cases, audit reports, or strategy. "
                "IMPORTANT: 'complaints' here means customer operational complaints — NOT legal filings."
            ),
        },
        {
            "agent_type": "genie",
            "genie_space": {"id": menu_genie_id},
            "name": "menu-analytics",
            "description": (
                "ONLY call for structured queries about menu items, nutrition, allergens, item pricing "
                "tiers, brand-level menu comparisons, or per-location compliance summaries. Keywords: "
                "calories, protein, fat, carbs, allergen-free, gluten-free, dairy-free, item price, "
                "brand menu, nutrition comparison, healthy options. Do NOT call for descriptive "
                "questions about individual dishes — use menu-document-search for those."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": inspect_ep_id},
            "name": "inspection-reports",
            "description": (
                "ONLY call for questions about specific food safety inspection documents, detailed "
                "inspector findings, corrective action details, or historical inspection report text. "
                "For inspection SCORES/GRADES use operations-intelligence instead."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": menu_ep_id},
            "name": "menu-document-search",
            "description": (
                "ONLY call for descriptive questions about specific dishes — ingredient details, "
                "preparation, dish description, or specific menu content. For nutrition stats, "
                "allergen filtering, or price comparisons, use menu-analytics instead."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": legal_ep_id},
            "name": "legal-complaints",
            "description": (
                "ONLY call for questions about lawsuits, legal cases, customer complaint filings, "
                "litigation status, legal liability, legal issues, or specific complaint case numbers. "
                "Always surface case numbers (format CK-XX-XXXX), risk levels (HIGH/MEDIUM/LOW), "
                "and amounts at stake."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": reg_ep_id},
            "name": "regulatory-compliance",
            "description": (
                "ONLY call for questions about permits, licenses, certifications, regulatory compliance "
                "documents, or specific regulatory requirements."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": audit_ep_id},
            "name": "audit-findings",
            "description": (
                "ONLY call for questions about financial or operational audit reports, audit findings, "
                "auditor recommendations, or specific audit references."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": consult_ep_id},
            "name": "consultancy-strategy",
            "description": (
                "ONLY call for questions about strategic recommendations, consultant advice, improvement "
                "strategies, or specific consultancy reports."
            ),
        },
    ],
    "instructions": _supervisor_instructions,
}

##### Register the supervisor instructions in the Prompt Registry

`SUPERVISOR_BODY["instructions"]` is the supervisor's "system prompt" — the
text the MAS API stores and replays at every routing call. Register it
under `{CATALOG}.prompts.supervisor_instructions` so it's versioned in UC
alongside the agent itself. Re-running the stage bumps the version and the
`production` alias.

In [ ]:
import sys
sys.path.append('../utils')
from prompt_registry import seed_prompt_history

# Two earlier versions of the supervisor instructions, seeded on first deploy
# so the Prompt Registry UI shows v1 → v2 → v3 history (not just the current
# version). The seed is NOT a real engineering changelog — it's synthetic
# demo history. seed_prompt_history tags each seeded version with
# is_demo_seed="true" so the audit trail is honest about that.
_SUPERVISOR_V1 = (
    "You are an operational AI assistant for Casper's Kitchens. "
    "Synthesise intelligence across the available agents to deliver insights to the executive team. "
    "Be concise and data-driven."
)
_SUPERVISOR_V2 = (
    "You are an elite operational AI assistant for Casper's Kitchens, a ghost kitchen network "
    "operating 16 restaurant brands across 8 locations (US: San Francisco, Silicon Valley, Bellevue, "
    "Chicago; EMEA: London, Munich, Amsterdam, Vianen). Synthesise intelligence from multiple "
    "sources to deliver concise, executive-ready insights. Always specify which location(s) you "
    "are referencing. Lead with the most important finding, then provide supporting detail. "
    "When discussing legal or audit matters, remind users to involve counsel or auditors for "
    "decisions. Be direct, data-driven, and strategic."
)

_common_tags = {
    "agent": "operational_supervisor",
    "stage": "operational_supervisor",
    "agent_name": AGENT_NAME,
    "consumed_via": "mlflow.genai.load_prompt at stage construction time",
}

seed_prompt_history(
    spark=spark,
    catalog=CATALOG,
    name="supervisor_instructions",
    historical=[
        {
            "template": _SUPERVISOR_V1,
            "commit_message": "v1: minimal synthesiser, no location or routing context (demo history seed)",
            "tags": _common_tags,
        },
        {
            "template": _SUPERVISOR_V2,
            "commit_message": "v2: added location context + executive framing (demo history seed)",
            "tags": _common_tags,
        },
    ],
    current={
        "template": _FALLBACK_SUPERVISOR_INSTRUCTIONS,
        "commit_message": f"{AGENT_NAME} routing/synthesis instructions (with explicit DATA SOURCE BOUNDARIES)",
        "tags": _common_tags,
    },
)

In [ ]:
def _list_mas():
    """List supervisor agents via the v2.1 SupervisorAgents API.

    The v2.0 path (API_BASE) only supports per-resource ops (POST/GET-by-id/PATCH/
    DELETE); listing on it returns 'No API found'.
    """
    mas, params = [], {}
    try:
        while True:
            resp = w.api_client.do("GET", "/api/2.1/supervisor-agents", query=params)
            mas.extend(resp.get("supervisor_agents", []))
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception as e:
        print(f"  Supervisor list error: {e}")
    return mas


def get_tile_by_name(name):
    try:
        params = {}
        while True:
            resp = w.api_client.do("GET", "/api/2.0/tiles", query=params)
            for tile in resp.get("tiles", []):
                if tile.get("name") == name:
                    return tile
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception:
        pass
    return None


agent_id = None
is_new = False

try:
    df = spark.sql(f"""
        SELECT resource_data FROM {CATALOG}._internal_state.resources
        WHERE resource_type = 'multi_agent_supervisors'
        ORDER BY created_at DESC
    """)
    for row in df.collect():
        info = json.loads(row.resource_data)
        if (info.get("endpoint_name") == SUPERVISOR_ENDPOINT or info.get("name") == AGENT_NAME) and info.get("tile_id"):
            agent_id = info["tile_id"]
            print(f"\u267b\ufe0f Supervisor found in uc_state ({agent_id}) — skipping create")
            break
except Exception as e:
    print(f"\u26a0\ufe0f uc_state lookup failed: {e}")

if not agent_id:
    for sa in _list_mas():
        # v2.1 items are flat dicts. `display_name` = user-facing name; `name` = resource path.
        if (
            sa.get("endpoint_name") == SUPERVISOR_ENDPOINT
            or sa.get("display_name") == AGENT_NAME
            or sa.get("name") == AGENT_NAME
        ):
            agent_id = sa.get("supervisor_agent_id") or sa.get("id") or sa.get("tile_id")
            print(f"\u267b\ufe0f Supervisor found in SupervisorAgents API ({agent_id}) — skipping create")
            break

if not agent_id:
    try:
        w.api_client.do("POST", API_BASE, body=SUPERVISOR_BODY)
        print("\u2705 Created Operational Supervisor")
        is_new = True
    except Exception as e:
        if "already exists" in str(e).lower():
            print("\u267b\ufe0f Supervisor already exists, proceeding")
        else:
            raise
    tile = get_tile_by_name(AGENT_NAME)
    if tile:
        agent_id = tile["tile_id"]
        print(f"Resolved tile_id: {agent_id}")
    else:
        agent_id = AGENT_NAME
        print(f"\u26a0\ufe0f Using name as fallback: {agent_id}")

if isinstance(agent_id, str) and len(agent_id) >= 8 and "-" in agent_id:
    actual_endpoint_name = f"mas-{agent_id[:8]}-endpoint"
else:
    actual_endpoint_name = SUPERVISOR_ENDPOINT
print(f"MAS serving endpoint name: {actual_endpoint_name}")

add(CATALOG, "multi_agent_supervisors", {
    "endpoint_name": actual_endpoint_name,
    "tile_id": agent_id,
    "name": AGENT_NAME,
})

# ── Drift recovery: PATCH the supervisor with fresh sub-agent IDs ────────────
# The `genie_spaces` stage is non-idempotent and creates fresh space IDs on
# every deploy. When this stage reuses an existing supervisor (uc_state hit
# above), its wired Genie IDs / KA endpoints can point at trashed resources,
# which surfaces in chat as "permissions issue accessing the revenue analytics
# system" (it's actually a 404 on the trashed Genie space). Re-applying
# SUPERVISOR_BODY against the existing tile id keeps the supervisor in lock-
# step with the current resources. No-op when nothing drifted.
#
# Hard failure on exhausted retries: this used to log a warning and continue,
# which left the supervisor pointing at trashed resources and broke the demo
# chat silently.  Now we retry the PATCH 3x with exponential backoff (1/2/4 s)
# and raise on the final failure so the deploy aborts with a clear error
# instead of leaving a half-wired supervisor in place.
import time as _time

if not is_new and isinstance(agent_id, str) and "-" in agent_id and len(agent_id) >= 8:
    existing = w.api_client.do("GET", f"{API_BASE}/{agent_id}")
    wired = (existing.get("multi_agent_supervisor") or {}).get("agents") or []

    def _key(a):
        t = a.get("agent_type", "")
        if t in ("genie", "genie-space"):
            return ("genie", a.get("name"), (a.get("genie_space") or {}).get("id"))
        if t in ("ka", "knowledge-assistant"):
            return ("ka", a.get("name"), (a.get("serving_endpoint") or {}).get("name"))
        return (t, a.get("name"))

    wired_keys  = {_key(a) for a in wired}
    target_keys = {_key(a) for a in SUPERVISOR_BODY["agents"]}

    # Also detect instructions drift — load_prompt above can load a newer
    # production version of the prompt that doesn't match what's currently
    # wired into the live supervisor. Without this check, a registry edit
    # would silently fail to take effect on already-existing supervisors.
    wired_instructions = ((existing.get("multi_agent_supervisor") or existing) or {}).get("instructions", "")
    target_instructions = SUPERVISOR_BODY["instructions"]
    instructions_drift = (wired_instructions or "") != (target_instructions or "")

    if wired_keys == target_keys and not instructions_drift:
        print("✅ Supervisor wiring + instructions match uc_state — no PATCH needed")
    else:
        if wired_keys != target_keys:
            drifted = target_keys ^ wired_keys
            print(f"⚠️  Supervisor wiring drift detected ({len(drifted)} mismatched sub-agents) — PATCHing…")
        if instructions_drift:
            print(f"⚠️  Supervisor instructions drift detected (load_prompt resolved a different version than the one wired in) — PATCHing…")
        # NOTE on body shape: the MAS PATCH API rejects partial bodies with
        # `Missing required field: name` even when update_mask is set, so we
        # send the whole SUPERVISOR_BODY. `examples` are a separate resource
        # at `/api/2.0/tiles/{id}/examples` — they are NOT touched by this
        # PATCH. Cell 15 below re-POSTs EXAMPLES on every run as a defensive
        # measure in case a prior deploy lost them.
        _last_err = None
        for _attempt in range(1, 4):
            try:
                w.api_client.do("PATCH", f"{API_BASE}/{agent_id}", body=SUPERVISOR_BODY)
                print(f"✅ Supervisor PATCHed with current Genie/KA wiring (attempt {_attempt}/3)")
                _last_err = None
                break
            except Exception as e:
                _last_err = e
                _wait = 2 ** (_attempt - 1)  # 1s, 2s, 4s
                print(f"⚠️  Drift-recovery PATCH attempt {_attempt}/3 failed: {e!r}")
                if _attempt < 3:
                    print(f"   retrying in {_wait}s…")
                    _time.sleep(_wait)
        if _last_err is not None:
            # Re-raise so the stage fails loudly.  A stale supervisor breaks
            # the demo chat ("permissions issue accessing the revenue
            # analytics system") in a way that's expensive to diagnose at
            # demo time — we'd much rather fail the deploy here.
            raise RuntimeError(
                f"Supervisor drift-recovery PATCH failed after 3 attempts: {_last_err!r}. "
                f"The supervisor at tile_id={agent_id} is still wired to stale Genie/KA "
                f"resources.  Re-run this stage once the Databricks API is reachable, or "
                f"delete the supervisor from uc_state to force re-creation."
            ) from _last_err

##### Attach evaluation examples (idempotent)

Re-attaches `EXAMPLES` to the supervisor on every run — defensive against any
prior whole-body PATCH that may have wiped them. POSTing the same examples is
safe (server replaces the list).

In [ ]:
# Examples attachment is IDEMPOTENT — re-attach on every run so a prior
# wipe/transient failure self-heals.  Correct API:
#   POST /api/2.0/multi-agent-supervisors/{tile_id}/examples
#   body: {"tile_id": <id>, "question": str, "guidelines": [str]}
# (one example per request — no batched payload).  The legacy
# /api/2.0/tiles/{id}/examples endpoint that was here before was wrong
# and silently failed inside the old try/except, which is why examples
# were not surviving deploys.
#
# Idempotency: best-effort list+delete before re-creating.  The list/delete
# endpoints may not exist on this surface — if either step 404s we just
# POST anyway, and accept that duplicates may accumulate across deploys.
#
# Robustness: small inter-request sleep + one retry on POST/DELETE failure.
# The MAS examples API rate-limits bursts of >~5 req/s and a fraction of POSTs
# come back with empty error bodies (the SDK stringifies those as `None`),
# so we slow the loop down and retry once with a backoff before giving up.

def _format_api_error(e: Exception) -> str:
    """str(e) on databricks-sdk errors is sometimes empty/None when the
    server returns a non-standard error body. Surface enough detail to debug."""
    msg = str(e) or ""
    if not msg or msg == "None":
        msg = repr(e)
    code = getattr(e, "error_code", None)
    return f"{type(e).__name__}: {msg}" + (f" [error_code={code}]" if code else "")


def _do_with_retry(method: str, path: str, body: dict | None = None, retries: int = 1, backoff_s: float = 2.0):
    """Call api_client.do with one retry on Exception. Returns the response or
    re-raises the last exception after the retries are exhausted."""
    last_exc = None
    for attempt in range(retries + 1):
        try:
            if body is None:
                return w.api_client.do(method, path)
            return w.api_client.do(method, path, body=body)
        except Exception as e:
            last_exc = e
            if attempt < retries:
                time.sleep(backoff_s)
    raise last_exc


_EXAMPLES_BASE = f"{API_BASE}/{agent_id}/examples"
if agent_id and agent_id != AGENT_NAME and actual_endpoint_name:
    if is_new:
        print(f"\nPolling MAS endpoint readiness ({actual_endpoint_name})...")
        ep_ready = False
        for attempt in range(1, 61):
            try:
                resp = w.api_client.do("GET", f"/api/2.0/serving-endpoints/{actual_endpoint_name}")
                state = resp.get("state", {})
                ready = str(state.get("ready", "")).upper()
                if ready == "READY":
                    print(f"  \u2705 Endpoint READY")
                    ep_ready = True
                    break
                print(f"  [{attempt}/60] state={ready} — waiting 20s…")
            except Exception as e:
                print(f"  [{attempt}/60] poll error: {e} — waiting 20s…")
            time.sleep(20)
    else:
        # Re-runs: assume endpoint is already READY (Readiness_Check stage
        # would have failed earlier otherwise).  Skip the 20-min poll loop.
        ep_ready = True

    if ep_ready:
        # Best-effort: clear existing examples before re-creating, so we
        # don't accumulate duplicates on every redeploy.  Skipped silently
        # if the list/delete endpoints aren't available on this surface.
        existing = []
        try:
            resp = w.api_client.do("GET", _EXAMPLES_BASE)
            existing = resp.get("examples", []) or []
        except Exception as _le:
            print(f"  (skipping pre-clear: {_format_api_error(_le)})")
        deleted, delete_failures = 0, 0
        for ex in existing:
            ex_id = ex.get("example_id") or ex.get("id")
            if not ex_id:
                continue
            try:
                _do_with_retry("DELETE", f"{_EXAMPLES_BASE}/{ex_id}", retries=1, backoff_s=2.0)
                deleted += 1
            except Exception as _de:
                delete_failures += 1
                print(f"  \u26a0\ufe0f  Could not delete existing example {ex_id}: {_format_api_error(_de)}")
            time.sleep(0.4)  # gentle pacing — MAS examples API rate-limits bursts
        if existing:
            print(f"  Cleared {deleted}/{len(existing)} existing example(s) before re-attach"
                  f"{f' (delete failures: {delete_failures})' if delete_failures else ''}")

        posted = 0
        failures = 0
        for ex in EXAMPLES:
            guideline = ex.get("guideline")
            guidelines = ex.get("guidelines")
            if guidelines is None and guideline is not None:
                guidelines = [guideline]
            body = {"tile_id": agent_id, "question": ex["question"]}
            if guidelines:
                body["guidelines"] = guidelines
            try:
                _do_with_retry("POST", _EXAMPLES_BASE, body=body, retries=1, backoff_s=2.0)
                posted += 1
            except Exception as _pe:
                failures += 1
                print(f"  \u26a0\ufe0f  Failed to attach example {ex['question']!r}: {_format_api_error(_pe)}")
            time.sleep(0.4)  # gentle pacing — MAS examples API rate-limits bursts
        if failures:
            print(f"\u26a0\ufe0f  Examples attached partially: {posted}/{len(EXAMPLES)} (failures={failures})")
        else:
            print(f"\u2705 Examples attached ({posted}/{len(EXAMPLES)} questions, is_new={is_new})")
    else:
        print("\u26a0\ufe0f  Endpoint not READY after 20 min — skipping examples (re-run to retry)")
else:
    print("\u26a0\ufe0f  Skipping examples — agent_id or endpoint not resolved")

print(f"\n\u2705 Operational Supervisor stage complete")
print(f"   Endpoint: {actual_endpoint_name}")
print("   Sub-agents: revenue-analytics, operations-intelligence, menu-analytics, "
      "inspection-reports, menu-document-search, legal-complaints, regulatory-compliance, "
      "audit-findings, consultancy-strategy")

##### Production monitoring scorers

Register the LLM-as-judge scorers that grade every live supervisor request.
Same idempotent register-or-restart pattern as `stages/refunder_agent.ipynb`
and `stages/complaint_agent.ipynb`, scoped to the supervisor's managed MLflow
experiment (`/Users/{me}/{mas_id}-dev-experiment`).

Doing this inside the stage — rather than relying solely on
`demos/operational-dashboard-demo/evaluation.ipynb` cell 27 — means the
supervisor always has judges immediately after deploy, even when the
downstream `Evaluation` task is skipped or fails partway through its
`mlflow.genai.evaluate()` calls.

5 scorers at 100% sampling:

- `safety` — built-in `Safety()` LLM judge for harmful or inappropriate content
- `relevance_to_query` — built-in `RelevanceToQuery()` — does the answer address the question
- `operational_quality` — generic `Guidelines` — concrete data, not a hedge
- `routing_accuracy` — custom `@scorer` — vocabulary signals consistent with expected sub-agent (skipped in prod when no `expected_agent` is provided — useful for eval-time, harmless in prod)
- `cites_specific_data` — custom `@scorer` — does the response cite numbers, IDs, dates rather than hedge

In [ ]:
import re

import mlflow
from mlflow.entities import Feedback
from mlflow.genai.scorers import (
    Guidelines,
    RelevanceToQuery,
    Safety,
    ScorerSamplingConfig,
    list_scorers,
    scorer,
)

# Resolve the supervisor's managed experiment.  Agent Bricks names the dev
# experiment "/Users/{me}/{mas_id}-dev-experiment" where mas_id is the
# serving-endpoint name with "-endpoint" stripped.  set_experiment creates
# the experiment if Agent Bricks hasn't materialised it yet — AB normally
# creates it lazily on the first prediction call, which hasn't happened
# yet at this point in the stage.
_me = spark.sql("SELECT current_user()").collect()[0][0]
_mas_id = actual_endpoint_name.replace("-endpoint", "")
sup_experiment_name = f"/Users/{_me}/{_mas_id}-dev-experiment"
sup_experiment = mlflow.set_experiment(sup_experiment_name)
sup_experiment_id = sup_experiment.experiment_id
print(f"Supervisor experiment: {sup_experiment_name} (id={sup_experiment_id})")


# Custom scorers — duplicated from demos/operational-dashboard-demo/evaluation.ipynb
# to keep this stage self-contained (no shared utils import from /utils — the
# stage already imports prompt_registry from there and adding more shared eval
# code muddles the separation).  Edit BOTH places if signatures change.
_AGENT_SIGNALS = {
    "revenue":     ["revenue", "cancellation rate", "order count", "orders placed", "sales", "avg order", "weekly", "total orders"],
    "operations":  ["complaint rate", "kitchen", "operational", "throughput", "busiest", "food safety grade", "cancel rate"],
    "inspection":  ["inspection report", "violation", "inspector", "corrective action", "grade", "score", "health permit"],
    "legal":       ["case no", "ck-", "risk level", "amount at stake", "counsel", "litigation", "plaintiff", "high risk"],
    "regulatory":  ["permit", "certificate", "expiry", "regulatory", "fda", "zoning", "issuing authority"],
    "audits":      ["audit", "auditor", "finding", "pwc", "deloitte", "kpmg", "significant", "critical finding"],
    "consultancy": ["consultant", "roi", "recommend", "strategy", "mckinsey", "phase 1", "investment"],
    "multi":       ["revenue", "legal", "audit", "operational"],
}


@scorer
def routing_accuracy(inputs: dict, outputs, expectations: dict = None) -> Feedback:
    """Heuristic: does the response use vocabulary consistent with the expected sub-agent?

    Returns None (skipped) when no `expected_agent` is provided — that's the
    norm in production traffic, where users don't tag their messages with the
    expected routing target.  Useful at eval time, harmless in prod.
    """
    def _as_text(o):
        # Inlined so the registered/serialized scorer can resolve it at runtime.
        if o is None: return ""
        if isinstance(o, str): return o
        if isinstance(o, dict):
            v = o.get("final_response")
            if isinstance(v, str): return v
            msgs = o.get("messages")
            if isinstance(msgs, list):
                for m in reversed(msgs):
                    if isinstance(m, dict) and m.get("role") == "assistant":
                        c = m.get("content")
                        if isinstance(c, str): return c
            choices = o.get("choices")
            if isinstance(choices, list) and choices:
                try:
                    c = choices[0]["message"]["content"]
                    if isinstance(c, str): return c
                except (KeyError, TypeError):
                    pass
            output = o.get("output")
            if isinstance(output, str): return output
            if isinstance(output, list):
                parts = []
                for it in output:
                    if isinstance(it, dict):
                        c = it.get("content")
                        if isinstance(c, str): parts.append(c)
                        elif isinstance(c, list):
                            for piece in c:
                                if isinstance(piece, dict):
                                    parts.append(piece.get("text", piece.get("value", "")) or "")
                if parts: return " ".join(parts)
        return str(o)
    expected_agent = (inputs or {}).get("expected_agent") or (expectations or {}).get("expected_agent", "")
    response_lower = _as_text(outputs).lower()
    if not expected_agent or expected_agent not in _AGENT_SIGNALS:
        return Feedback(value=None, rationale="No expected_agent specified — skipped")
    signals = _AGENT_SIGNALS[expected_agent]
    if expected_agent == "multi":
        domains_present = sum(
            1 for ds in _AGENT_SIGNALS.values()
            if any(s in response_lower for s in ds)
        )
        score = min(1.0, domains_present / 4.0)
        rationale = f"{domains_present} agent domains detected in board summary response"
    else:
        matched = [s for s in signals if s in response_lower]
        score = min(1.0, len(matched) / max(2, len(signals) // 2))
        rationale = f"Matched signals: {matched}" if matched else "No expected-agent signals detected in response"
    return Feedback(value=score, rationale=rationale)


@scorer
def cites_specific_data(inputs: dict, outputs, expectations: dict = None) -> Feedback:
    """Heuristic: did the response cite concrete data (numbers, IDs, dates) rather than hedge?"""
    import re  # imported inside the function so the registered/serialized scorer can resolve it at runtime
    def _as_text(o):
        # Inlined so the registered/serialized scorer can resolve it at runtime.
        # Eval time hands us the string from predict_fn; prod monitoring hands us
        # the raw trace outputs (dict-shaped for the MAS endpoint).
        if o is None: return ""
        if isinstance(o, str): return o
        if isinstance(o, dict):
            v = o.get("final_response")
            if isinstance(v, str): return v
            msgs = o.get("messages")
            if isinstance(msgs, list):
                for m in reversed(msgs):
                    if isinstance(m, dict) and m.get("role") == "assistant":
                        c = m.get("content")
                        if isinstance(c, str): return c
            choices = o.get("choices")
            if isinstance(choices, list) and choices:
                try:
                    c = choices[0]["message"]["content"]
                    if isinstance(c, str): return c
                except (KeyError, TypeError):
                    pass
            output = o.get("output")
            if isinstance(output, str): return output
            if isinstance(output, list):
                parts = []
                for it in output:
                    if isinstance(it, dict):
                        c = it.get("content")
                        if isinstance(c, str): parts.append(c)
                        elif isinstance(c, list):
                            for piece in c:
                                if isinstance(piece, dict):
                                    parts.append(piece.get("text", piece.get("value", "")) or "")
                if parts: return " ".join(parts)
        return str(o)
    response = _as_text(outputs)
    patterns = [
        r"\d+\.?\d*\s*%",                                # percentages
        r"\$\s*[\d,]+",                                   # dollar amounts
        r"CK-\d+-\d+",                                    # legal case numbers
        r"\d{4}-\d{2}-\d{2}",                             # ISO dates
        r"\b(?:score|grade)\s*(?:of\s*)?\d+",             # inspection scores
        r"\b\d+x\b",                                      # ROI multiples
        r"\b\d+\s+(?:cases?|orders?|locations?|findings?)",
    ]
    found = [p for p in patterns if re.search(p, response, re.IGNORECASE)]
    hedges = ["i don't have access", "i cannot provide", "i'm unable", "consult your", "please check with"]
    is_hedged = any(h in response.lower() for h in hedges)
    if is_hedged and len(found) < 2:
        return Feedback(value=0.0, rationale="Response is hedged with no concrete data — agent likely failed to retrieve")
    score = min(1.0, len(found) / 3.0)
    return Feedback(value=score, rationale=f"Data patterns found: {len(found)} types — {found}")


def _register_scorer(scorer_obj, name: str, sample_rate: float = 1.0):
    """Idempotent register-or-restart, scoped to the supervisor's prod experiment.

    Re-running the stage hits the same code path; calling .register() on an
    already-registered scorer raises ValueError, so we look it up first and
    just .start() it instead.  Same shape as stages/refunder_agent.ipynb and
    stages/complaint_agent.ipynb.
    """
    existing = {s.name: s for s in list_scorers(experiment_id=sup_experiment_id)}
    sampling = ScorerSamplingConfig(sample_rate=sample_rate)
    if name in existing:
        existing[name].start(sampling_config=sampling)
        print(f"  ↺ {name} — restarted at {sample_rate:.0%} sample rate")
        return existing[name]
    registered = scorer_obj.register(name=name, experiment_id=sup_experiment_id)
    registered.start(sampling_config=sampling)
    print(f"  ✅ {name} — registered + started at {sample_rate:.0%} sample rate")
    return registered


# Baseline — every agent in the bundle gets these three.
_register_scorer(Safety(),           name="safety",             sample_rate=1.0)
_register_scorer(RelevanceToQuery(), name="relevance_to_query", sample_rate=1.0)
_register_scorer(
    Guidelines(
        name="operational_quality",
        guidelines=(
            "The response must include specific data points such as numbers, percentages, "
            "dates, case numbers, or named locations. "
            "The response must not be a generic hedge or refusal (e.g. 'I don't have access'). "
            "The response must directly answer the question asked."
        ),
    ),
    name="operational_quality",
    sample_rate=1.0,
)

# Domain — supervisor-specific routing and data-citation heuristics.
_register_scorer(routing_accuracy,    name="routing_accuracy",    sample_rate=1.0)
_register_scorer(cites_specific_data, name="cites_specific_data", sample_rate=1.0)

print("\n✅ Production monitoring enabled — 5 scorers active at 100% sampling")
print(f"   View judges:    {sup_experiment_name} → Monitoring tab")